## Modeling Phase

### Introduction
------------------------------------------------

In this phase, I build machine learning models that predict whether a Titanic passenger survived. The goal is not only to generate predictions, but also to understand which factors influence survival.

Following the CRISP-DM [2] framework, this phase includes the following steps:
 - Select appropriate modeling techniques
 - Define the test design
 - Train the candidate models
 - Prepare model outputs for evaluation
 - Save trained models for the next phase

The models are trained using the prepared dataset from the previous phase.

In [1]:
import pandas as pd
import numpy as np
from titanic_surv.dataset import load_processed_data

# Load training dataset
training_set = load_processed_data("titanic_training.csv")
validating_set = load_processed_data("titanic_validate.csv")

2026-03-13 18:33:02.403 | INFO     | titanic_surv.config:<module>:11 - PROJ_ROOT path is: D:\py\Titanic\titanic_surv_pred_insights


### 1. Select Modelling Technique
------------------------------------------------

My goal here is to choose models that can predict survival and are still easy to interpret.

Since the target variable Survived is binary (0 = did not survive, 1 = survived), I focus on models that are commonly used for classification problems.

**Candidate Models**
 - **Logistic Regression** - A simple linear model used for binary classification. The model produces coefficients that show how each feature affects the survival probability.
 - **Decision Tree** - A rule-based model that splits data using decision rules. The structure is easy to explain and visualize.
 - **Random Forest** - An ensemble model that combines many decision trees. This usually produces more stable predictions than a single tree.

These models are implemented using scikit-learn, which is a widely used Python library for machine learning.

**Why These Models**

Using interpretable models helps explain why certain passengers survived or did not survive. This supports the goal of generating insights rather than only producing predictions.

Previous machine learning studies also recommend these models for classification tasks:
 - *Logistic Regression* is often used as a baseline for binary outcomes [4].
 - *Decision Trees* are transparent and easy to explain [5].
 - *Random Forest* improves prediction stability by combining multiple trees [6].

Because the dataset is class-imbalanced, I will use class weighting during training to reduce bias toward the majority class [1], [4].

#### 1.1 Justification for Model Selection from Related Work

In this project, I selected **Logistic Regression**, **Decision Tree**, and **Random Forest** because recent Titanic studies repeatedly use these models and report useful performance with similar passenger features.

- **Huang (2024)** shows that Random Forest can perform strongly for Titanic survival prediction [7].
- **Devamane (2024)** uses Logistic Regression and Random Forest with key features such as age, sex, and passenger class, which is consistent with this project design [8].
- **Singh and Nagpal** compare multiple classifiers, including Logistic Regression and Decision Tree, and support a compare-first approach instead of choosing one model too early [9].

Based on this, my model setup follows two goals at the same time:

### 2. Generate Test Design
------------------------------------------------

Before training the models, I define a consistent experimental setup. This ensures that each model is trained and evaluated under the same conditions.

The dataset was already divided during the Data Preparation phase into three parts:
 - Training Set (70%) – used to train the models
 - Validation Set (15%) – used to generate predictions for comparison
 - Test Set (15%) – reserved for final evaluation in Phase 5

In this phase:
 - I train the models using the training dataset.
 - I generate predictions on the validation dataset.
 - I store the prediction outputs for the next phase.

I will not select the best model or tune decision thresholds here. That analysis will happen in Phase 5 – Evaluation.

#### Feature Setup

The models use the following prepared features:
 - Age
 - Fare
 - SibSp
 - Parch
 - Family_Size
 - Sex_Code
 - Pclass_Code
The target variable is Survived.

These features were selected because earlier analysis showed they contain meaningful survival patterns.


In [2]:
# Define features and target
model_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "Family_Size",
    "Sex_Code",
    "Pclass_Code",
]

target_column = "Survived"

# Training set
train_x = training_set[model_features]
train_y = training_set[target_column]

# Validation set
val_x = validating_set[model_features]
val_y = validating_set[target_column]

print("Feature and target sets are ready for modeling.")
print(f"Training: {train_x.shape}")
print(f"Validation shape: {val_x.shape}")

Feature and target sets are ready for modeling.
Training: (916, 7)
Validation shape: (196, 7)


**Modeling Research Questions:** 

This phase focuses on answering the following questions:
1. **RQ1** - Can Logistic Regression, Decision Tree, and Random Forest all be trained successfully using the prepared feature set? [4],[6]
2. **RQ2** - Do these models generate complete and valid prediction outputs on the validation dataset? [3]
3. **RQ3** - Does this modeling workflow create a fair and reproducible setup for comparing models in the next phase? [3],[6]

### 3. Building the Model
------------------------------------------------

In this step, I train the three candidate models using the same dataset and feature set. This ensures that the comparison between models is fair.

The models trained are:
 - Logistic Regression
 - Decision Tree
 - Random Forest

All models are trained using the training dataset and then used to generate predictions on the validation dataset.

#### Step 1: Import model libraries
Import the necessary libraries from sklearn so I can build and train all three models.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

####  Step 2: Train all candidate models

To handle class imbalance, I set the parameter:
```powershell
class_weight = "balanced"
```

This tells the algorithm to give more importance to the minority class (survivors), which helps reduce bias toward the majority class [1].

Using class weighting is also practical because it keeps the original dataset unchanged and works consistently across different models [1], [4].

In [4]:
# Define models with balanced class weights to handle potential class imbalance
models = {
    "Logistic Regression": LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    )
}

# Set up dictionaries to store predictions and probabilities
preds = {}
probs = {}

# Loop through each model, fit it, and store predictions and probabilities
for name, model in models.items():
    model.fit(train_x, train_y)
    preds[name] = model.predict(val_x)
    probs[name] = model.predict_proba(val_x)[:, 1] if hasattr(model, "predict_proba") else None
    print(f"{name} training completed.")

Logistic Regression training completed.
Decision Tree training completed.
Random Forest training completed.


#### Step 3: Training Results

All three models trained successfully using:
 - the same training dataset
 - the same target variable (Survived)
 - the same feature set

This confirms that the modeling workflow is consistent.

**Answer to RQ1:**

Yes. Logistic Regression, Decision Tree, and Random Forest can all be trained successfully using the prepared feature set.

#### Step 4: Regularization in Logistic Regression

Logistic Regression includes regularization by default.

Regularization helps prevent overfitting by penalizing very large model coefficients.

In this implementation:

 - LogisticRegression(C=1.0)
 - Uses L2 regularization, where the parameter C controls the strength of the penalty. Smaller values of C apply stronger regularization.

Tree-based models such as Decision Tree and Random Forest do not use coefficient penalties. Instead, they control model complexity through parameters such as:
 - max_depth
 - min_samples_leaf
 - n_estimators

These parameters limit how complex the trees can grow.

### 4. Prepare model outputs for Evaluation phase
------------------------------------------------

After training the models, I prepare their outputs so they can be analyzed in the next phase.

For each model, I store:
 - predicted class labels (y_pred)
 - predicted survival probabilities (y_prob)
 - the true validation labels (y_true)
 
I also run checks to confirm that the outputs are valid.

These checks verify that:
 - the number of predictions matches the validation dataset size
 - predicted labels are binary (0 or 1)
 - probability outputs are available

In [5]:
rows = []
validation_outputs = {}

for name in models.keys():
    y_pred = preds[name]
    y_prob = probs[name]

    pred_len_ok = len(y_pred) == len(val_y)
    is_binary = set(pd.Series(y_pred).unique()).issubset({0, 1})

    validation_outputs[name] = {
        "y_true": val_y.to_numpy(),
        "y_pred": y_pred,
        "y_prob": y_prob
    }

    rows.append({
        "Model": name,
        "Validation_Rows": len(y_pred),
        "Prediction_Length_OK": pred_len_ok,
        "Binary_Predictions_OK": is_binary,
        "Has_Probability_Output": y_prob is not None
    })

results_df = pd.DataFrame(rows)
display(results_df)
print("Validation prediction artifacts are prepared for Phase 5 evaluation.")

,Model,Validation_Rows,Prediction_Length_OK,Binary_Predictions_OK,Has_Probability_Output
0,Logistic Regression,196,True,True,True
1,Decision Tree,196,True,True,True
2,Random Forest,196,True,True,True


Validation prediction artifacts are prepared for Phase 5 evaluation.


**Insight:**  
- All models passed these checks.

**Answer to RQ2:**
*Yes*. All models generated valid prediction outputs for the validation dataset.

### 5. Save trained models
------------------------------------------------

Finally, I save the trained models so they can be reused later without retraining.

Each model is saved as a .pkl file, and the validation prediction outputs are also saved.

Saving these files supports reproducibility because the same models and prediction results can be loaded again in the evaluation phase.

This also allows consistent analysis of model confidence, fairness, and error patterns.

**Answer to RQ3:**

Yes. The Phase 4 workflow provides a fair and reproducible setup for comparing models in the evaluation phase.

In [6]:
import joblib

joblib.dump(models["Logistic Regression"], "../models/logistic_ml_model.pkl")
joblib.dump(models["Decision Tree"], "../models/decision_tree_ml_model.pkl")
joblib.dump(models["Random Forest"], "../models/random_forest_ml_model.pkl")

# Save validation prediction artifacts for Phase 5 evaluation
joblib.dump(validation_outputs, "../models/validation_outputs.pkl")

print("All candidate models and validation artifacts are saved for Phase 5.")

All candidate models and validation artifacts are saved for Phase 5.


### References
------------------------------------------------

[1] H. He and E. A. Garcia, "Learning from imbalanced data," *IEEE Transactions on Knowledge and Data Engineering*, vol. 21, no. 9, pp. 1263-1284, 2009.

[2] Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). CRISP-DM 1.0: Step-by-step data mining guide.

[3] Scikit-learn Developers, "Classification metrics and threshold tuning," *scikit-learn documentation*. [Online]. Available: https://scikit-learn.org/

[4] G. James, D. Witten, T. Hastie, and R. Tibshirani, *An Introduction to Statistical Learning: With Applications in R*, 2nd ed. New York, NY, USA: Springer, 2021.

[5] L. Breiman, J. H. Friedman, R. A. Olshen, and C. J. Stone, *Classification and Regression Trees*. Belmont, CA, USA: Wadsworth, 1984.

[6] L. Breiman, "Random forests," *Machine Learning*, vol. 45, no. 1, pp. 5-32, 2001.

[7] Y. Huang, "Comparative analysis of models based on Titanic survival prediction," 2024.

[8] A. Devamane, "Titanic survivors prediction analysis using machine learning," 2024.

[9] A. Singh and A. Nagpal, "Exploratory data analysis and machine learning on Titanic data."